In [2]:
import requests

client_id = "3acd117d-0ebb-4623-ac3a-e4d144a8868f"
search_text = "østfold"  # County or region
element = "air_pressure_at_sea_level"

endpoint_sources = "https://frost.met.no/sources/v0.jsonld"
params = {
    "types": "SensorSystem",
    "county": search_text,
    "elements": element
}
r = requests.get(endpoint_sources, params=params, auth=(client_id, ""))
r.raise_for_status()
data = r.json()

stations = []
for station in data.get("data", []):
    stations.append({"id": station["id"], "name": station["name"]})

print(f"Stations in '{search_text}' measuring '{element}':")
for s in stations:
    print(f"{s['id']}: {s['name']}")

Stations in 'østfold' measuring 'air_pressure_at_sea_level':
SN17150: RYGGE
SN17000: STRØMTANGEN FYR
SN17280: GULLHOLMEN


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import requests

terrain_level = 23.77
installation_depth = 6.8


# --- Read Absoluttryck data ---
pvt_path = r"\\nsv2-nasuni-02\fredrikstad\Prosjekt\O10245\10245026-01\10245026-01-03_ARBEIDSOMRAADE\21_fagomraade\11_Geoteknikk\10245026-07-FELT-OG LABREGISTRERINGER\Poretrykksmålinger\Avlesninger\28.11.2023\viken2pvt_32767_20231128_025807.pvt"
pvt_path_1 = r"\\nsv2-nasuni-02\fredrikstad\Prosjekt\O10245\10245026-01\10245026-01-03_ARBEIDSOMRAADE\21_fagomraade\11_Geoteknikk\10245026-07-FELT-OG LABREGISTRERINGER\Poretrykksmålinger\Avlesninger\28.11.2023\viken2pvt_32132_20231128_030021.pvt"

with open(pvt_path, encoding="cp1252") as f:
    for i, line in enumerate(f):
        if line.strip().startswith("Datum") or line.strip().startswith("Date"):
            header_row = i
            break

df = pd.read_csv(pvt_path, sep="\t", skiprows=header_row, encoding="cp1252")
df = df.dropna(axis=1, how='all')
# Normalize column names to handle both Swedish and English headers
column_mapping = {
    'Date': 'Datum',
    'Time': 'Klockslag',
    'Absolute pressure': 'Absoluttryck',
    'Temperature': 'Temperatur',
    'Battery': 'Batteri'
}
df.rename(columns=column_mapping, inplace=True)
df = df[df['Datum'].notna() & (df['Datum'].str.strip() != '')]
df['datetime'] = pd.to_datetime(df['Datum'] + ' ' + df['Klockslag'], format='%Y-%m-%d %H:%M', errors='coerce')
# Handle both comma and period as decimal separator
df['Absoluttryck'] = df['Absoluttryck'].astype(str).str.replace(',', '.', regex=False)
df['Absoluttryck'] = pd.to_numeric(df['Absoluttryck'], errors='coerce')
df = df.sort_values('datetime')
df['date'] = df['datetime'].dt.date  # For merging with daily API data

# --- Fetch precipitation and air pressure from Frost API ---
client_id = "3acd117d-0ebb-4623-ac3a-e4d144a8868f"  # Replace with your Frost API client ID
station_id = "SN17000"        # Example: sarpsborg

start = df['datetime'].min().strftime('%Y-%m-%d')
end = df['datetime'].max().strftime('%Y-%m-%d')

endpoint = "https://frost.met.no/observations/v0.jsonld"
params = {
    "sources": station_id,
    "elements": "sum(precipitation_amount P1D),mean(air_pressure_at_sea_level P1D)",
    "referencetime": f"{start}/{end}"
}
r = requests.get(endpoint, params=params, auth=(client_id, ""))
r.raise_for_status()
data = r.json()

# Parse precipitation and air pressure data
rain = []
for item in data.get("data", []):
    date = item["referenceTime"][:10]
    precip = None
    lufttrykk = None
    for obs in item["observations"]:
        if obs["elementId"] == "sum(precipitation_amount P1D)":
            precip = obs.get("value")
        elif obs["elementId"] == "mean(air_pressure_at_sea_level P1D)":
            lufttrykk = obs.get("value")
    rain.append({"date": date, "rain_mm": precip, "lufttrykk": lufttrykk})

rain_df = pd.DataFrame(rain)
rain_df["date"] = pd.to_datetime(rain_df["date"]).dt.date

# --- Merge and calculate corrected pressure ---
df = pd.merge(df, rain_df, on="date", how="left")
# Convert lufttrykk from hPa to mH2O (1 hPa ≈ 0.0101972 mH2O)
df["absoluttrykk_kPa"] = df["Absoluttryck"] /10194*100000  # Convert mH2O to kPa

# df["lufttrykk_mH2O"] = df["lufttrykk"] * 0.0101972
df["Absoluttryck_korr"] = df["absoluttrykk_kPa"] - df["lufttrykk"] * 100 / 1000

# --- Plot ---
fig = go.Figure()

# Corrected Absoluttryck (left y-axis)
fig.add_trace(go.Scatter(
    x=df['datetime'],
    y=df['Absoluttryck_korr'],
    name="Korrigert trykk (kPa)",
    mode="lines",
    yaxis="y1"
))

# Rainfall (right y-axis, bars)
fig.add_trace(go.Bar(
    x=pd.to_datetime(rain_df["date"]),
    y=rain_df["rain_mm"],
    name="Nedbør (mm)",
    yaxis="y2",
    opacity=0.5,
    marker_color='blue'
))

fig.add_trace(go.Scatter(
    x=(df['datetime'].iloc[0],df['datetime'].iloc[-1]),
    y=(terrain_level, terrain_level), 
    name="Terrengnivå [m]", mode="lines", line=dict(dash='dash', color='green')
))

fig.update_layout(
    title="Korrigert trykk og Nedbør",
    xaxis=dict(title="Tid"),
    yaxis=dict(title="Korrigert trykk (kPa)", side="left"),
    yaxis2=dict(
        title="Nedbør (mm)",
        overlaying="y",
        side="right",
        showgrid=False
    ),
    legend=dict(x=0.01, y=0.99)
)

fig.show()

In [9]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests

# --- Inputs ---
terrain_level = 31.36
installation_depth = 15
water_level = terrain_level - 5
gamma_w = 9.81  # kN/m³


# --- Read Absoluttryck data ---
pvt_path = r"\\nsv2-nasuni-02\fredrikstad\Prosjekt\O10245\10245026-01\10245026-01-03_ARBEIDSOMRAADE\21_fagomraade\11_Geoteknikk\10245026-07-FELT-OG LABREGISTRERINGER\Poretrykksmålinger\Avlesninger\28.11.2023\viken2pvt_32767_20231128_025807.pvt"
pvt_path_1 = r"\\nsv2-nasuni-02\fredrikstad\Prosjekt\O10245\10245026-01\10245026-01-03_ARBEIDSOMRAADE\21_fagomraade\11_Geoteknikk\10245026-07-FELT-OG LABREGISTRERINGER\Poretrykksmålinger\Avlesninger\28.11.2023\viken2pvt_32132_20231128_030021.pvt"
pvt_path_2 = r"\\nsv2-nasuni-02\fredrikstad\Prosjekt\O10245\10245026-01\10245026-01-03_ARBEIDSOMRAADE\21_fagomraade\11_Geoteknikk\10245026-07-FELT-OG LABREGISTRERINGER\Poretrykksmålinger\Avlesninger\20.11.23\sarp201123_32151_20231120_115410.pvt"

with open(pvt_path_2, encoding="cp1252") as f:
    for i, line in enumerate(f):
        if line.strip().startswith("Datum") or line.strip().startswith("Date"):
            header_row = i
            break
            

df = pd.read_csv(pvt_path_2, sep="\t", skiprows=header_row, encoding="cp1252")
df = df.dropna(axis=1, how='all')
# Normalize column names to handle both Swedish and English headers
column_mapping = {
    'Date': 'Datum',
    'Time': 'Klockslag',
    'Absolute pressure': 'Absoluttryck',
    'Temperature': 'Temperatur',
    'Battery': 'Batteri'
}
df.rename(columns=column_mapping, inplace=True)
df = df[df['Datum'].notna() & (df['Datum'].str.strip() != '')]
df['datetime'] = pd.to_datetime(df['Datum'] + ' ' + df['Klockslag'], format='%Y-%m-%d %H:%M', errors='coerce')
# Handle both comma and period as decimal separator
df['Absoluttryck'] = df['Absoluttryck'].astype(str).str.replace(',', '.', regex=False)
df['Absoluttryck'] = pd.to_numeric(df['Absoluttryck'], errors='coerce')
df = df.sort_values('datetime')
df['date'] = df['datetime'].dt.date  # For merging with daily API data

print(f"Read {len(df)} rows from PVT file")
print(f"Date range: {df['datetime'].min()} to {df['datetime'].max()}")

# --- Fetch precipitation and air pressure from Frost API ---
client_id = "3acd117d-0ebb-4623-ac3a-e4d144a8868f"  # Replace with your Frost API client ID
station_id = "SN17000"        # Example: sarpsborg

start = df['datetime'].min().strftime('%Y-%m-%d')
end = df['datetime'].max().strftime('%Y-%m-%d')

print(f"Fetching Frost API data from {start} to {end}")

endpoint = "https://frost.met.no/observations/v0.jsonld"
params = {
    "sources": station_id,
    "elements": "sum(precipitation_amount P1D),mean(air_pressure_at_sea_level P1D)",
    "referencetime": f"{start}/{end}"
}
r = requests.get(endpoint, params=params, auth=(client_id, ""))
r.raise_for_status()
data = r.json()

# Parse precipitation and air pressure data
rain = []
for item in data.get("data", []):
    date = item["referenceTime"][:10]
    precip = None
    lufttrykk = None
    for obs in item["observations"]:
        if obs["elementId"] == "sum(precipitation_amount P1D)":
            precip = obs.get("value")
        elif obs["elementId"] == "mean(air_pressure_at_sea_level P1D)":
            lufttrykk = obs.get("value")
    rain.append({"date": date, "rain_mm": precip, "lufttrykk": lufttrykk})

rain_df = pd.DataFrame(rain)
rain_df["date"] = pd.to_datetime(rain_df["date"]).dt.date

print(f"Fetched {len(rain_df)} days of weather data")
print(f"Days with air pressure data: {rain_df['lufttrykk'].notna().sum()}")

# --- Merge and calculate corrected pressure ---
df = pd.merge(df, rain_df, on="date", how="left")
print(f"After merge: {len(df)} rows, {df['lufttrykk'].notna().sum()} with air pressure data")

# Convert lufttrykk from hPa to mH2O (1 hPa ≈ 0.0101972 mH2O)
df["absoluttrykk_kPa"] = df["Absoluttryck"] /10194*100000  # Convert mH2O to kPa

# df["lufttrykk_mH2O"] = df["lufttrykk"] * 0.0101972
df["Absoluttryck_korr"] = df["absoluttrykk_kPa"] - df["lufttrykk"] * 100 / 1000


# --- Prepare heights for hydrostatic profile ---
hydro_y = np.array([water_level, terrain_level - installation_depth])
hydro_y = hydro_y[hydro_y <= water_level]

# --- Hydrostatic pressure profile ---
hydrostatic_pressures = (water_level - hydro_y) * gamma_w

# --- Prepare measured/corrected pore pressure profile ---
df_valid = df.dropna(subset=["Absoluttryck", "lufttrykk"]).copy()

if len(df_valid) == 0:
    print("\n" + "="*60)
    print("ERROR: No valid data after filtering!")
    print("="*60)
    print(f"Total rows in dataframe: {len(df)}")
    print(f"Rows with Absoluttryck: {df['Absoluttryck'].notna().sum()}")
    print(f"Rows with lufttrykk: {df['lufttrykk'].notna().sum()}")
    print("\nSample of data:")
    print(df[['Datum', 'Klockslag', 'Absoluttryck', 'date', 'lufttrykk']].head())
    raise ValueError("No valid data to plot. Check if Frost API returned air pressure data for your date range.")

df_valid["absoluttrykk_kPa"] = df_valid["Absoluttryck"] / 10194 * 100000
df_valid["Absoluttrykk_korr"] = df_valid["absoluttrykk_kPa"] - df_valid["lufttrykk"] * 100 / 1000

pore_y = np.full(len(df_valid), terrain_level - installation_depth)
pore_x = df_valid["Absoluttrykk_korr"].values

print(f"Valid data points for plotting: {len(pore_x)}")

# --- 90th percentile ---
percentile_90 = np.percentile(pore_x, 90)
measurement_depth = terrain_level - installation_depth
y_cross = measurement_depth + percentile_90 / gamma_w
print(f"Hydrostatisk linje fra 90-persentil krysser x=0 ved y = {y_cross:.2f} moh")
# --- Hydrostatic line from 90th percentile up to water level ---
# The hydrostatic line: p = (water_level - y) * gamma_w
# We want a line that starts at (percentile_90, measurement_depth) and follows the hydrostatic gradient up to water_level
hydrostat_y = np.linspace(measurement_depth, y_cross, 50)
hydrostat_x = percentile_90 + (water_level - hydrostat_y) * gamma_w - (water_level - measurement_depth) * gamma_w
# Hydrostatic pressure at measurement depth
hydrostatic_at_depth = (water_level - measurement_depth) * gamma_w

# Percentage of 90-percentile pressure vs hydrostatic
percent_of_hydrostatic = 100 * percentile_90 / hydrostatic_at_depth if hydrostatic_at_depth != 0 else np.nan
print(f"90-persentil utgjør {percent_of_hydrostatic:.1f}% av hydrostatisk trykk ved måledypet.")

# --- Plot ---
fig = go.Figure()

# Hydrostatic pressure line (from water level down)
fig.add_trace(go.Scatter(
    x=hydrostatic_pressures,
    y=hydro_y,
    mode='lines+markers',
    name='Hydrostatisk trykk',
    line=dict(color='blue', dash='dash')
))

# All measured/corrected pore pressures (as points)
fig.add_trace(go.Scatter(
    x=pore_x,
    y=pore_y,
    mode='markers+lines',
    name='Korrigert målt poretrykk',
    marker=dict(color='red', size=10, symbol='circle'),
    line=dict(color='red', width=1, dash='dot')
))

# 90th percentile marker
fig.add_trace(go.Scatter(
    x=[percentile_90],
    y=[measurement_depth],
    mode='markers',
    name='90-persentil',
    marker=dict(color='yellow', size=16, symbol='diamond')
))

# Hydrostatic line from 90th percentile, following hydrostatic gradient
fig.add_trace(go.Scatter(
    x=hydrostat_x,
    y=hydrostat_y,
    mode='lines',
    name='Hydrostatisk linje fra 90-persentil',
    line=dict(color='blue', dash='dot', width=3)
))

# Terrain level line (horizontal)
fig.add_trace(go.Scatter(
    x=[0, max(list(hydrostatic_pressures) + list(pore_x) + [percentile_90])*1.1],
    y=[terrain_level, terrain_level],
    mode='lines',
    name='Terrengnivå',
    line=dict(color="black", dash="dot")
))

# Water level line (horizontal)
fig.add_trace(go.Scatter(
    x=[0, max(list(hydrostatic_pressures) + list(pore_x) + [percentile_90])*1.1],
    y=[water_level, water_level],
    mode='lines',
    name='Vannstand',
    line=dict(color="cyan", dash="dot")
))

fig.update_layout(
    title="Poretrykkprofil (korrigert og hydrostatisk)",
    xaxis_title="Trykk (kPa)",
    yaxis_title="Høyde (moh)"
)
fig.add_trace(go.Scatter(
    x=[0, percentile_90],
    y=[water_level, measurement_depth],
    mode='lines',
    name='Linje grunnvann→90-persentil',
    line=dict(color='green', dash='dash', width=2)
))

fig.show()

Read 273 rows from PVT file
Date range: 2023-02-21 00:21:00 to 2023-11-20 00:21:00
Fetching Frost API data from 2023-02-21 to 2023-11-20
Fetched 272 days of weather data
Days with air pressure data: 272
After merge: 273 rows, 272 with air pressure data
Valid data points for plotting: 272
Hydrostatisk linje fra 90-persentil krysser x=0 ved y = 26.42 moh
90-persentil utgjør 100.6% av hydrostatisk trykk ved måledypet.


In [8]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np
import requests

terrain_level = 31.36
installation_depth_1 = 7
installation_depth_2 = 15
water_level = terrain_level - 5
gamma_w = 9.81  # kN/m³

# --- Read Absoluttryck data for both files ---
pvt_path = r"\\nsv2-nasuni-02\fredrikstad\Prosjekt\O10245\10245026-01\10245026-01-03_ARBEIDSOMRAADE\21_fagomraade\11_Geoteknikk\10245026-07-FELT-OG LABREGISTRERINGER\Poretrykksmålinger\Avlesninger\28.11.2023\viken2pvt_32767_20231128_025807.pvt"
pvt_path_1 = r"\\nsv2-nasuni-02\fredrikstad\Prosjekt\O10245\10245026-01\10245026-01-03_ARBEIDSOMRAADE\21_fagomraade\11_Geoteknikk\10245026-07-FELT-OG LABREGISTRERINGER\Poretrykksmålinger\Avlesninger\28.11.2023\viken2pvt_32132_20231128_030021.pvt"
SB_2003_pvt_hyd = r"\\nsv2-nasuni-02\fredrikstad\Prosjekt\O10245\10245026-01\10245026-01-03_ARBEIDSOMRAADE\21_fagomraade\11_Geoteknikk\10245026-07-FELT-OG LABREGISTRERINGER\Poretrykksmålinger\Avlesninger\20.11.23\SB_2003_hydraulisk.pvt"
SB_2003_pvt_el = r"\\nsv2-nasuni-02\fredrikstad\Prosjekt\O10245\10245026-01\10245026-01-03_ARBEIDSOMRAADE\21_fagomraade\11_Geoteknikk\10245026-07-FELT-OG LABREGISTRERINGER\Poretrykksmålinger\Avlesninger\20.11.23\sarp201123_32151_20231120_115410.pvt"
def read_pvt(path):
    with open(path, encoding="cp1252") as f:
        for i, line in enumerate(f):
            if line.strip().startswith("Datum") or line.strip().startswith("Date"):
                header_row = i
                break
    df = pd.read_csv(path, sep="\t", skiprows=header_row, encoding="cp1252")
    df = df.dropna(axis=1, how='all')
    # Normalize column names to handle both Swedish and English headers
    column_mapping = {
        'Date': 'Datum',
        'Time': 'Klockslag',
        'Absolute pressure': 'Absoluttryck',
        'Temperature': 'Temperatur',
        'Battery': 'Batteri'
    }
    df.rename(columns=column_mapping, inplace=True)
    df = df[df['Datum'].notna() & (df['Datum'].str.strip() != '')]
    df['datetime'] = pd.to_datetime(df['Datum'] + ' ' + df['Klockslag'], format='%Y-%m-%d %H:%M', errors='coerce')
    # Handle both comma and period as decimal separator
    df['Absoluttryck'] = df['Absoluttryck'].astype(str).str.replace(',', '.', regex=False)
    df['Absoluttryck'] = pd.to_numeric(df['Absoluttryck'], errors='coerce')
    df = df.sort_values('datetime')
    df['date'] = df['datetime'].dt.date
    return df

df = read_pvt(SB_2003_pvt_hyd)
df1 = read_pvt(SB_2003_pvt_el)

# --- Fetch precipitation and air pressure from Frost API ---
client_id = "3acd117d-0ebb-4623-ac3a-e4d144a8868f"
station_id = "SN17000"  # Example: sarpsborg

start = min(df['datetime'].min(), df1['datetime'].min()).strftime('%Y-%m-%d')
end = max(df['datetime'].max(), df1['datetime'].max()).strftime('%Y-%m-%d')

endpoint = "https://frost.met.no/observations/v0.jsonld"
params = {
    "sources": station_id,
    "elements": "sum(precipitation_amount P1D),mean(air_pressure_at_sea_level P1D)",
    "referencetime": f"{start}/{end}"
}
r = requests.get(endpoint, params=params, auth=(client_id, ""))
r.raise_for_status()
data = r.json()

# Parse precipitation and air pressure data
rain = []
for item in data.get("data", []):
    date = item["referenceTime"][:10]
    precip = None
    lufttrykk = None
    for obs in item["observations"]:
        if obs["elementId"] == "sum(precipitation_amount P1D)":
            precip = obs.get("value")
        elif obs["elementId"] == "mean(air_pressure_at_sea_level P1D)":
            lufttrykk = obs.get("value")
    rain.append({"date": date, "rain_mm": precip, "lufttrykk": lufttrykk})

rain_df = pd.DataFrame(rain)
rain_df["date"] = pd.to_datetime(rain_df["date"]).dt.date

# --- Merge and calculate corrected pressure for both datasets ---
def merge_and_correct(df, rain_df, is_corrected=True):
    if is_corrected:
        df = pd.merge(df, rain_df, on="date", how="left")
        df["absoluttrykk_kPa"] = df["Absoluttryck"] / 10194 * 100000
        df["Absoluttrykk_korr"] = df["absoluttrykk_kPa"] - df["lufttrykk"] * 100 / 1000
    else:
        df = pd.merge(df, rain_df, on="date", how="left")
        df["Absoluttrykk_korr"] = df["Absoluttryck"]

    return df

df = merge_and_correct(df, rain_df, is_corrected=False)
df1 = merge_and_correct(df1, rain_df)

# --- Function to generate all traces for a given water level ---
def generate_traces_for_water_level(wl, df_valid, df1_valid, terrain_level, installation_depth_1, installation_depth_2, gamma_w):
    traces = []
    annotations = []
    
    # Hydrostatic reference lines
    hydro_y = np.array([wl, terrain_level - max(installation_depth_1, installation_depth_2)])
    hydro_y = hydro_y[hydro_y <= wl]
    hydrostatic_pressures = (wl - hydro_y) * gamma_w
    
    # Plot for first dataset
    measurement_depth_1 = terrain_level - installation_depth_1
    pore_y = np.full(len(df_valid), measurement_depth_1)
    pore_x = df_valid["Absoluttrykk_korr"].values
    
    percentile_90 = np.percentile(pore_x, 90)
    y_cross = measurement_depth_1 + percentile_90 / gamma_w
    hydrostatic_at_depth = (wl - measurement_depth_1) * gamma_w
    percent_of_hydrostatic = 100 * percentile_90 / hydrostatic_at_depth if hydrostatic_at_depth != 0 else np.nan
    
    hydrostat_y = np.linspace(measurement_depth_1, y_cross, 50)
    hydrostat_x = percentile_90 + (wl - hydrostat_y) * gamma_w - (wl - measurement_depth_1) * gamma_w
    
    traces.append(go.Scatter(x=hydrostatic_pressures, y=hydro_y, mode='lines+markers', 
                            name='Hydrostatisk trykk', line=dict(color='blue')))
    traces.append(go.Scatter(x=pore_x, y=pore_y, mode='markers+lines', 
                            name='Korrigert målt poretrykk', 
                            marker=dict(color='orange', size=10, symbol='circle'),
                            line=dict(color='orange', width=1, dash='dot')))
    traces.append(go.Scatter(x=[percentile_90], y=[measurement_depth_1], mode='markers',
                            name='90-persentil', marker=dict(color='gray', size=16, symbol='diamond')))
    traces.append(go.Scatter(x=hydrostat_x, y=hydrostat_y, mode='lines',
                            name='Hydrostatisk linje fra 90-persentil',
                            line=dict(color='blue', dash='dot', width=3)))
    traces.append(go.Scatter(x=[0, percentile_90], y=[wl, measurement_depth_1], mode='lines',
                            name='Linje grunnvann→90-persentil',
                            line=dict(color='blue', dash='dash', width=2)))
    
    # Add annotation for first sensor
    annotations.append(dict(
        x=percentile_90,
        y=measurement_depth_1,
        text=f"Krysser x=0 ved y={y_cross:.2f}m<br>{percent_of_hydrostatic:.1f}% av hydrostatisk",
        showarrow=True,
        arrowhead=2,
        ax=40,
        ay=-40,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="gray",
        borderwidth=1
    ))
    
    # Plot for second dataset
    measurement_depth_2 = terrain_level - installation_depth_2
    pore1_y = np.full(len(df1_valid), measurement_depth_2)
    pore1_x = df1_valid["Absoluttrykk_korr"].values
    
    percentile1_90 = np.percentile(pore1_x, 90)
    y1_cross = measurement_depth_2 + percentile1_90 / gamma_w
    hydrostatic1_at_depth = (wl - measurement_depth_2) * gamma_w
    percent1_of_hydrostatic = 100 * percentile1_90 / hydrostatic1_at_depth if hydrostatic1_at_depth != 0 else np.nan
    
    hydrostat1_y = np.linspace(measurement_depth_2, y1_cross, 50)
    hydrostat1_x = percentile1_90 + (wl - hydrostat1_y) * gamma_w - (wl - measurement_depth_2) * gamma_w
    
    traces.append(go.Scatter(x=pore1_x, y=pore1_y, mode='markers+lines',
                            name='Korrigert målt poretrykk (2)',
                            marker=dict(color='orange', size=10, symbol='circle'),
                            line=dict(color='orange', width=1, dash='dot')))
    traces.append(go.Scatter(x=[percentile1_90], y=[measurement_depth_2], mode='markers',
                            name='90-persentil (2)', marker=dict(color='gray', size=16, symbol='diamond')))
    traces.append(go.Scatter(x=hydrostat1_x, y=hydrostat1_y, mode='lines',
                            name='Hydrostatisk linje fra 90-persentil (2)',
                            line=dict(color='blue', dash='dot', width=3)))
    traces.append(go.Scatter(x=[0, percentile1_90], y=[wl, measurement_depth_2], mode='lines',
                            name='Linje grunnvann→90-persentil (2)',
                            line=dict(color='blue', dash='dash', width=2)))
    
    # Add annotation for second sensor
    annotations.append(dict(
        x=percentile1_90,
        y=measurement_depth_2,
        text=f"Krysser x=0 ved y={y1_cross:.2f}m<br>{percent1_of_hydrostatic:.1f}% av hydrostatisk",
        showarrow=True,
        arrowhead=2,
        ax=40,
        ay=40,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="gray",
        borderwidth=1
    ))
    
    # Terrain and water level lines
    max_x = max(list(hydrostatic_pressures) + list(pore_x) + [percentile_90] +
                list(pore1_x) + [percentile1_90]) * 1.1
    
    traces.append(go.Scatter(x=[0, max_x], y=[terrain_level, terrain_level], mode='lines',
                            name='Terrengnivå', line=dict(color="black")))
    traces.append(go.Scatter(x=[0, max_x], y=[wl, wl], mode='lines',
                            name='Vannstand', line=dict(color="blue")))
    
    return traces, annotations

# --- Prepare valid data once ---
df_valid = df.dropna(subset=["Absoluttryck", "lufttrykk"]).copy()
df1_valid = df1.dropna(subset=["Absoluttryck", "lufttrykk"]).copy()

# --- Generate water level range: -2 to +2 meters in 0.25m steps ---
water_levels = np.arange(water_level - 2, water_level + 2.01, 0.25)
initial_wl_index = int(len(water_levels) / 2)  # Middle value (original water_level)

# --- Generate all trace sets ---
all_trace_sets = []
all_annotation_sets = []
for wl in water_levels:
    traces, annotations = generate_traces_for_water_level(wl, df_valid, df1_valid, terrain_level, 
                                            installation_depth_1, installation_depth_2, gamma_w)
    all_trace_sets.append(traces)
    all_annotation_sets.append(annotations)

# --- Create figure with initial water level ---
fig = go.Figure(data=all_trace_sets[initial_wl_index])

# --- Create slider steps ---
slider_steps = []
for i, wl in enumerate(water_levels):
    # For each step, we need to update all trace data and annotations
    step = {
        'args': [
            {'x': [trace.x for trace in all_trace_sets[i]],
             'y': [trace.y for trace in all_trace_sets[i]]},
            {'title': f'Poretrykkprofil - Vannstand: {wl:.2f} m',
             'annotations': all_annotation_sets[i]}
        ],
        'label': f'{wl:.2f}',
        'method': 'update'
    }
    slider_steps.append(step)

fig.update_layout(
    title=f"Poretrykkprofil - Vannstand: {water_level:.2f} m",
    xaxis_title="Trykk (kPa)",   
    yaxis_title="Høyde (moh)",
    width=800,
    annotations=all_annotation_sets[initial_wl_index],
    sliders=[{
        'active': initial_wl_index,
        'yanchor': 'top',
        'y': -0.15,
        'xanchor': 'left',
        'currentvalue': {
            'prefix': 'Vannstand: ',
            'suffix': ' m',
            'visible': True,
            'xanchor': 'center',
            'font': {'size': 14}
        },
        'pad': {'b': 10, 't': 50},
        'len': 0.9,
        'x': 0.05,
        'steps': slider_steps
    }]
)

fig.show()

In [20]:
# -*- coding: utf-8 -*-
import requests
import pandas as pd
# import package_installer as pi
# pi.import_or_install('plotly', '6.3.0')
# # Try to install kaleido without version constraint as it can be picky
# pi.import_or_install('kaleido', '1.2.0')
import plotly.graph_objects as go
from pathlib import Path


def get_available_counties(element: str) -> list:
    """Fetch available counties from Frost API."""
    r = requests.get(
        "https://frost.met.no/sources/v0.jsonld",
        params={
            "types": "SensorSystem",
            "elements": element
        },
        auth=("3acd117d-0ebb-4623-ac3a-e4d144a8868f", ""),
        timeout=5
    )
    r.raise_for_status()
    data = r.json().get("data", [])
    return sorted({
        s["county"]
        for s in data
        if s.get("county")
    })


def get_weather_stations(county: str, element: str) -> list:
    """Fetch weather stations for a given county."""
    r = requests.get(
        "https://frost.met.no/sources/v0.jsonld",
        params={
            "types": "SensorSystem",
            "county": county.lower(),
            "elements": element
        },
        auth=("3acd117d-0ebb-4623-ac3a-e4d144a8868f", ""),
        timeout=5
    )
    r.raise_for_status()
    data = r.json().get("data", [])
    return [
        f"{s['id']} – {s['name']}"
        for s in data
        if "id" in s and "name" in s
    ]

def read_pvt_file(path):
    """Read and parse PVT file."""
    with open(path, encoding="cp1252") as f:
        for i, line in enumerate(f):
            if line.strip().startswith("Datum") or line.strip().startswith("Date"):
                header_row = i
                break
    df = pd.read_csv(path, sep="\t", skiprows=header_row, encoding="cp1252")
    df = df.dropna(axis=1, how='all')
    # Normalize column names to handle both Swedish and English headers
    column_mapping = {
        'Date': 'Datum',
        'Time': 'Klockslag',
        'Absolute pressure': 'Absoluttryck',
        'Temperature': 'Temperatur',
        'Battery': 'Batteri'
    }
    df.rename(columns=column_mapping, inplace=True)
    df = df[df['Datum'].notna() & (df['Datum'].str.strip() != '')]
    df['datetime'] = pd.to_datetime(df['Datum'] + ' ' + df['Klockslag'], format='%Y-%m-%d %H:%M', errors='coerce')
    # Handle both comma and period as decimal separator
    df['Absoluttryck'] = df['Absoluttryck'].astype(str).str.replace(',', '.', regex=False)
    df['Absoluttryck'] = pd.to_numeric(df['Absoluttryck'], errors='coerce')
    df = df.sort_values('datetime')
    df['date'] = df['datetime'].dt.date
    return df

def read_excel_file(path, serial):
    """Read and parse Excel file with multiple sheets, selecting by serial number.
    
    Args:
        path: Path to Excel file
        serial: Serial number to identify which sheet to read
    
    Returns:
        DataFrame with processed data
    """
    # Read Excel file with the sheet named after the serial number
    df = pd.read_excel(path, sheet_name=str(serial), header=None)
    
    # Find the header row (look for row with "Serial" AND other data columns)
    header_row = None
    for i in range(min(30, len(df))):  # Check first 30 rows
        row_values = df.iloc[i].astype(str).str.strip().str.lower()
        row_values_original = df.iloc[i].astype(str).str.strip()
        
        # Check if this row contains typical column headers
        # Look for combinations that indicate this is the header row
        contains_serial = any('serial' in val for val in row_values)
        contains_datetime = any(val in ['date time', 'date', 'datum'] for val in row_values)
        contains_pressure = any(val in ['mmh2o', 'kpa', 'absoluttryck', 'absolute pressure'] for val in row_values)
        
        if contains_serial and (contains_datetime or contains_pressure):
            header_row = i
            break
    
    # Re-read with proper header
    if header_row is not None:
        df = pd.read_excel(path, sheet_name=str(serial), header=header_row)
    else:
        raise ValueError(f"Could not find header row with data columns in sheet {serial}. First 10 rows:\n{df.head(10).to_string()}")
    
    df = df.dropna(axis=1, how='all')
    
    # Drop the Serial column if it exists (we already know which sheet we're reading)
    if 'Serial' in df.columns:
        df = df.drop(columns=['Serial'])
    
    # Track which pressure unit we have for proper conversion later
    # Check before renaming to avoid duplicate columns
    pressure_unit = None
    pressure_col = None
    if 'mmH2O' in df.columns:
        pressure_unit = 'mmH2O'
        pressure_col = 'mmH2O'
    elif 'mH2O' in df.columns:
        pressure_unit = 'mH2O'
        pressure_col = 'mH2O'
    elif 'kPa' in df.columns:
        pressure_unit = 'kPa'
        pressure_col = 'kPa'
    
    # Normalize column names to handle different formats
    # Only map the pressure column that exists
    column_mapping = {
        'Date': 'Datum',
        'Time': 'Klockslag',
        'Date Time': 'datetime_str',  # Combined date time column
        'Absolute pressure': 'Absoluttryck',
        'Temperature': 'Temperatur',
        '°C': 'Temperatur',
        'Battery': 'Batteri',
        'Volt': 'Batteri'
    }
    
    # Add the pressure column to mapping if found
    if pressure_col:
        column_mapping[pressure_col] = 'Absoluttryck'
    
    df.rename(columns=column_mapping, inplace=True)
    
    # Handle datetime - check if we have combined datetime or separate date/time
    if 'datetime_str' in df.columns:
        # Combined Date Time column
        df['datetime'] = pd.to_datetime(df['datetime_str'], errors='coerce')
    elif 'Datum' in df.columns and 'Klockslag' in df.columns:
        # Separate Date and Time columns
        df['datetime'] = pd.to_datetime(df['Datum'].astype(str) + ' ' + df['Klockslag'].astype(str), 
                                         format='%Y-%m-%d %H:%M', errors='coerce')
    else:
        raise ValueError(f"Could not find datetime columns. Available columns: {df.columns.tolist()}")
    
    # Filter out empty rows
    df = df[df['datetime'].notna()]
    
    # Handle pressure - convert to mH2O (same unit as PVT files)
    if 'Absoluttryck' in df.columns:
        # Handle both comma and period as decimal separator
        df['Absoluttryck'] = df['Absoluttryck'].astype(str).str.replace(',', '.', regex=False)
        df['Absoluttryck'] = pd.to_numeric(df['Absoluttryck'], errors='coerce')
        
        # Convert based on original unit to mH2O
        if pressure_unit == 'mmH2O':
            # millimeters H2O to meters H2O: divide by 1000
            df['Absoluttryck'] = df['Absoluttryck'] / 1000
        elif pressure_unit == 'mH2O':
            # already in meters H2O, no conversion needed
            pass
        elif pressure_unit == 'kPa':
            # kPa to mH2O: 1 kPa ≈ 0.102 mH2O (or divide by 9.81)
            df['Absoluttryck'] = df['Absoluttryck'] / 9.81
        # If no unit detected, assume it's already in mH2O
    
    df = df.sort_values('datetime')
    df['date'] = df['datetime'].dt.date
    
    return df

def fetch_weather_data(station_id: str, start: str, end: str) -> pd.DataFrame:
    """Fetch weather data from Frost API."""
    endpoint = "https://frost.met.no/observations/v0.jsonld"
    params = {
        "sources": station_id,
        "elements": "sum(precipitation_amount P1D),mean(air_pressure_at_sea_level P1D)",
        "referencetime": f"{start}/{end}"
    }
    r = requests.get(
        endpoint, 
        params=params, 
        auth=("3acd117d-0ebb-4623-ac3a-e4d144a8868f", "")
    )
    r.raise_for_status()
    data = r.json()

    rain = []
    for item in data.get("data", []):
        date = item["referenceTime"][:10]
        precip = None
        lufttrykk = None
        for obs in item["observations"]:
            if obs["elementId"] == "sum(precipitation_amount P1D)":
                precip = obs.get("value")
            elif obs["elementId"] == "mean(air_pressure_at_sea_level P1D)":
                lufttrykk = obs.get("value")
        rain.append({"date": date, "rain_mm": precip, "lufttrykk": lufttrykk})

    rain_df = pd.DataFrame(rain)
    rain_df["date"] = pd.to_datetime(rain_df["date"]).dt.date
    return rain_df


def calculate_pressure(df: pd.DataFrame, rain_df: pd.DataFrame, measurement_level: float) -> pd.DataFrame:
    """Calculate corrected pressure and trykkhøyde."""
    df = pd.merge(df, rain_df, on="date", how="left")
    df["absoluttrykk_kPa"] = df["Absoluttryck"] / 10194 * 100000
    df["Absoluttryck_korr"] = df["absoluttrykk_kPa"] - df["lufttrykk"] * 100 / 1000
    df["Trykkhøyde"] = df["Absoluttryck_korr"] * 1000 / 9.81 / 1000 + measurement_level
    return df


def create_plot(df: pd.DataFrame, rain_df: pd.DataFrame, terrain_level: float, borepoint: str, measurement_level: float, installation_depth: float, station_id: str, serial_number: str) -> go.Figure:
    """Create plotly figure for poretrykk data."""
    fig = go.Figure()

    max_trykkhoyde = df['Trykkhøyde'].max()
    min_trykkhoyde = df['Trykkhøyde'].min()
    max_idx = df['Trykkhøyde'].idxmax()
    min_idx = df['Trykkhøyde'].idxmin()
    mean_trykkhoyde = df['Trykkhøyde'].mean()
    mean_idx = (df['Trykkhøyde'] - mean_trykkhoyde).abs().idxmin()

    fig.add_trace(go.Scatter(
        x=df['datetime'],
        y=df['Trykkhøyde'],
        name="Trykkhøyde (m)",
        mode="lines",
        line=dict(dash='solid', color='crimson'),
        yaxis="y1"
    ))

    fig.add_trace(go.Bar(
        x=pd.to_datetime(rain_df["date"]),
        y=rain_df["rain_mm"],
        name="Nedbør (mm)",
        yaxis="y2",
        opacity=0.5,
        marker_color='blue'
    ))

    fig.add_trace(go.Scatter(
        x=(df['datetime'].iloc[0], df['datetime'].iloc[-1]),
        y=(terrain_level, terrain_level),
        name="Terrengnivå [m]",
        mode="lines",
        line=dict(dash='dash', color='black')
    ))

    fig.add_trace(go.Scatter(
        x=[df.loc[max_idx, 'datetime']],
        y=[max_trykkhoyde],
        mode='markers',
        name=f'Maks: {max_trykkhoyde:.1f} m',
        marker=dict(color='red', size=12, symbol='triangle-up')
    ))

    fig.add_trace(go.Scatter(
        x=[df.loc[min_idx, 'datetime']],
        y=[min_trykkhoyde],
        mode='markers',
        name=f'Min: {min_trykkhoyde:.1f} m',
        marker=dict(color='green', size=12, symbol='triangle-down')
    ))

    fig.add_trace(go.Scatter(
        x=[df.loc[mean_idx, 'datetime']],
        y=[mean_trykkhoyde],
        mode='markers',
        name=f'Gj.snitt: {mean_trykkhoyde:.1f} m',
        marker=dict(color='rgba(255, 255, 255, 0.8)', size=0, symbol='circle')
    ))
    # Create info box text
    info_text = (
        f"<b>Måleinformasjon:</b><br>"
        f"Terrengnivå: {terrain_level:.1f} m<br>"
        f"Installasjonsdybde: {installation_depth:.1f} m<br>"
        f"Målenivå: {measurement_level:.1f} m<br>"
        f"Værstasjon: {station_id}<br>"
        f"Lufttrykkskorrigert: Ja"
    )

    fig.add_annotation(
        xref="paper", yref="paper",
        x=0,  # Left edge (0 to 1 scale)
        y=-0.075, # Below the legend
        text=info_text,
        showarrow=False,
        align="left",
        bgcolor="rgba(255, 255, 255, 0.9)",
        # bordercolor="black",
        # borderwidth=1,
        borderpad=10,
        font=dict(size=11),
        xanchor="left",
        yanchor="top"
    )

    fig.update_layout(
        title=dict(
            xref="paper", yref="paper",
            text=f"{borepoint} - Poretrykksregistering - Målernr. {serial_number}",
            x=0,
            xanchor='left'
        ),
        xaxis=dict(
            title="Dato (-)", 
            tickformat="%d.%m.%Y",
            showline=True,
            linewidth=2,
            linecolor='grey',
            mirror=True
        ),
        yaxis=dict(
            title="Kotenivå (m)", 
            side="left",
            showline=True,
            linewidth=2,
            linecolor='grey',
            mirror=True,
            showgrid=True,
            gridcolor='lightgrey',
            griddash='dash'
        ),
        yaxis2=dict(
            title="Nedbør (mm)",
            overlaying="y",
            side="right",
            showgrid=False,
            title_standoff=25,
            ticks="inside",
            ticklen=5,
            ticklabelstandoff=15,
            automargin=True
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0,
            bgcolor="rgba(255, 255, 255, 0.8)",
            
        ),
        plot_bgcolor='white',
        paper_bgcolor='white',
        margin=dict(b=175, r=120, l=120, t=150),
        width=1123,   # A4 landscape width in pixels
        height=794    # A4 landscape height in pixels
    )
    return fig

def process_poretrykk(
    pvt_path: str,
    serial_number: str,
    station_id: str,
    terrain_level: float,
    installation_depth: float,
    borepoint: str
) -> str:
    """Main processing function."""
    measurement_level = terrain_level - installation_depth

    # Determine file type and read data accordingly
    if pvt_path.lower().endswith('.xlsx') or pvt_path.lower().endswith('.xls'):
        # Excel file - requires serial number
        if not serial_number:
            raise ValueError("Serial number is required for Excel files")
        df = read_excel_file(pvt_path, serial_number)
    else:
        # PVT file
        df = read_pvt_file(pvt_path)

    # Fetch weather data
    start = df['datetime'].min().strftime('%Y-%m-%d')
    end = df['datetime'].max().strftime('%Y-%m-%d')
    rain_df = fetch_weather_data(station_id, start, end)

    # Calculate pressure
    df = calculate_pressure(df, rain_df, measurement_level)

    # Create plot
    fig = create_plot(df, rain_df, terrain_level, borepoint, measurement_level, installation_depth, station_id, serial_number)

    # Save figure
    output_dir = Path(pvt_path).parent
    output_file_html = output_dir / f"{borepoint}_poretrykk_plot.html"
    output_file_pdf = output_dir / f"{borepoint}_poretrykk_plot.pdf"
    
    fig.write_html(str(output_file_html))
    fig.show()
    # Try to save as PDF
    try:
        import arcpy
        has_arcpy = True
    except ImportError:
        has_arcpy = False
    
    try:
        # Attempt PDF generation (kaleido is used automatically)
        # scale parameter helps avoid "Kaleido-fier" header artifact
        fig.write_image(str(output_file_pdf), width=1123, height=794, scale=1)
        
        # Check if PDF was actually created
        if output_file_pdf.exists():
            if has_arcpy:
                arcpy.AddMessage(f"PDF saved to: {output_file_pdf}")
        else:
            if has_arcpy:
                arcpy.AddWarning("PDF generation reported success but file not found")
    except Exception as e:
        # PDF generation failed
        if has_arcpy:
            arcpy.AddWarning(f"PDF generation failed: {str(e)}")
            arcpy.AddMessage("HTML file is available as alternative")

    return str(output_file_html)


def process_poretrykksprofil(
    pvt_path_1: str,
    serial_number_1: str,
    pvt_path_2: str,
    serial_number_2: str,
    station_id: str,
    terrain_level: float,
    installation_depth_1: float,
    installation_depth_2: float,
    water_depth_below_terrain: float,
    borepoint_nr: str,
) -> str:
    """Process one or two pore pressure sensors to create a profile plot with water level slider."""
    import numpy as np
    
    gamma_w = 9.81  # kN/m³
    water_level = terrain_level - water_depth_below_terrain
    
    # Check if second file is provided
    has_second_sensor = pvt_path_2 and pvt_path_2.strip() != ''
    
    # Read first dataset
    if pvt_path_1.lower().endswith(('.xlsx', '.xls')):
        df1 = read_excel_file(pvt_path_1, serial_number_1)
    else:
        df1 = read_pvt_file(pvt_path_1)
    
    # Read second dataset if provided
    df2 = None
    df2_valid = None
    measurement_level_2 = None
    if has_second_sensor:
        if pvt_path_2.lower().endswith(('.xlsx', '.xls')):
            df2 = read_excel_file(pvt_path_2, serial_number_2)
        else:
            df2 = read_pvt_file(pvt_path_2)
    
    # Fetch weather data
    if has_second_sensor:
        start = min(df1['datetime'].min(), df2['datetime'].min()).strftime('%Y-%m-%d')
        end = max(df1['datetime'].max(), df2['datetime'].max()).strftime('%Y-%m-%d')
    else:
        start = df1['datetime'].min().strftime('%Y-%m-%d')
        end = df1['datetime'].max().strftime('%Y-%m-%d')
    rain_df = fetch_weather_data(station_id, start, end)
    
    # Calculate corrected pressure for datasets
    measurement_level_1 = terrain_level - installation_depth_1
    df1 = calculate_pressure(df1, rain_df, measurement_level_1)
    df1_valid = df1.dropna(subset=["Trykkhøyde"]).copy()
    
    if has_second_sensor:
        measurement_level_2 = terrain_level - installation_depth_2
        df2 = calculate_pressure(df2, rain_df, measurement_level_2)
        df2_valid = df2.dropna(subset=["Trykkhøyde"]).copy()
    
    # Generate water level range: -2 to +2 meters in 0.25m steps
    water_levels = np.arange(water_level - 2, water_level + 2.01, 0.25)
    initial_wl_index = int(len(water_levels) / 2)
    
    # Function to generate traces for a given water level
    def generate_traces(wl):
        traces = []
        annotations = []
        
        # Hydrostatic reference line
        max_depth = installation_depth_2 if has_second_sensor else installation_depth_1
        hydro_y = np.array([wl, terrain_level - max_depth])
        hydro_y = hydro_y[hydro_y <= wl]
        hydrostatic_pressures = (wl - hydro_y) * gamma_w
        
        # First sensor data
        pore_x_1 = (df1_valid["Trykkhøyde"] - measurement_level_1) * gamma_w
        pore_y_1 = np.full(len(df1_valid), measurement_level_1)
        percentile_90_1 = np.percentile(pore_x_1, 90)
        y_cross_1 = measurement_level_1 + percentile_90_1 / gamma_w
        hydrostatic_at_depth_1 = (wl - measurement_level_1) * gamma_w
        percent_of_hydrostatic_1 = 100 * percentile_90_1 / hydrostatic_at_depth_1 if hydrostatic_at_depth_1 != 0 else 0
        
        hydrostat_y_1 = np.linspace(measurement_level_1, y_cross_1, 50)
        hydrostat_x_1 = percentile_90_1 + (wl - hydrostat_y_1) * gamma_w - hydrostatic_at_depth_1
        
        # Add traces for first sensor
        traces.append(go.Scatter(x=hydrostatic_pressures, y=hydro_y, mode='lines+markers', 
                                name='Hydrostatisk trykk', line=dict(color='blue')))
        traces.append(go.Scatter(x=pore_x_1, y=pore_y_1, mode='markers+lines', 
                                name=f'{serial_number_1} - Målt poretrykk', 
                                marker=dict(color='grey', size=8, symbol='circle'),
                                line=dict(color='grey', width=1, dash='dot')))
        traces.append(go.Scatter(x=[percentile_90_1], y=[measurement_level_1], mode='markers',
                                name=f'{serial_number_1} - 90-persentil', marker=dict(color='darkblue', size=14, symbol='diamond')))
        traces.append(go.Scatter(x=hydrostat_x_1, y=hydrostat_y_1, mode='lines',
                                name=f'{serial_number_1} - Hydrostatisk linje',
                                line=dict(color='blue', dash='dot', width=2)))
        traces.append(go.Scatter(x=[0, percentile_90_1], y=[wl, measurement_level_1], mode='lines',
                                name=f'{serial_number_1} - Linje grunnvann→90-persentil',
                                line=dict(color='blue', dash='dash', width=2)))
        
        # Annotations for first sensor
        annotations.append(dict(
            x=percentile_90_1, y=measurement_level_1,
            text=f"Hydrostatisk linje krysser x=0 ved y={y_cross_1:.2f}m<br>{percent_of_hydrostatic_1:.1f}% av hydrostatisk",
            showarrow=True, arrowhead=2, ax=110, ay=-40,
            bgcolor="rgba(255,255,255,0.9)", bordercolor="gray", borderwidth=1
        ))
        
        # Second sensor data (if provided)
        if has_second_sensor:
            pore_x_2 = (df2_valid["Trykkhøyde"] - measurement_level_2) * gamma_w
            pore_y_2 = np.full(len(df2_valid), measurement_level_2)
            percentile_90_2 = np.percentile(pore_x_2, 90)
            y_cross_2 = measurement_level_2 + percentile_90_2 / gamma_w
            hydrostatic_at_depth_2 = (wl - measurement_level_2) * gamma_w
            percent_of_hydrostatic_2 = 100 * percentile_90_2 / hydrostatic_at_depth_2 if hydrostatic_at_depth_2 != 0 else 0
            
            hydrostat_y_2 = np.linspace(measurement_level_2, y_cross_2, 50)
            hydrostat_x_2 = percentile_90_2 + (wl - hydrostat_y_2) * gamma_w - hydrostatic_at_depth_2
            
            traces.append(go.Scatter(x=pore_x_2, y=pore_y_2, mode='markers+lines',
                                    name=f'{serial_number_2} - Målt poretrykk',
                                    marker=dict(color='grey', size=8, symbol='circle'),
                                    line=dict(color='grey', width=1, dash='dot')))
            traces.append(go.Scatter(x=[percentile_90_2], y=[measurement_level_2], mode='markers',
                                    name=f'{serial_number_2} - 90-persentil', marker=dict(color='darkblue', size=14, symbol='diamond')))
            traces.append(go.Scatter(x=hydrostat_x_2, y=hydrostat_y_2, mode='lines',
                                    name=f'{serial_number_2} - Hydrostatisk linje',
                                    line=dict(color='blue', dash='dot', width=2)))
            traces.append(go.Scatter(x=[0, percentile_90_2], y=[wl, measurement_level_2], mode='lines',
                                    name=f'{serial_number_2} - Linje grunnvann→90-persentil',
                                    line=dict(color='blue', dash='dash', width=2)))
            
            annotations.append(dict(
                x=percentile_90_2, y=measurement_level_2,
                text=f"Hydrostatisk linje krysser x=0 ved y={y_cross_2:.2f}m<br>{percent_of_hydrostatic_2:.1f}% av hydrostatisk",
                showarrow=True, arrowhead=2, ax=0, ay=40,
                bgcolor="rgba(255,255,255,0.9)", bordercolor="gray", borderwidth=1
            ))
            max_x = max([percentile_90_1, percentile_90_2]) * 1.2
        else:
            max_x = percentile_90_1 * 1.2
        
        
        # Terrain and water level lines
        traces.append(go.Scatter(x=[0, max_x], y=[terrain_level, terrain_level], mode='lines',
                                name='Terrengnivå', line=dict(color="black", width=2, dash='dash')))
        traces.append(go.Scatter(x=[0, max_x], y=[wl, wl], mode='lines',
                                name='Vannstand', line=dict(color="blue", width=2)))
        
        return traces, annotations
    
    # Generate all trace sets for slider
    all_trace_sets = []
    all_annotation_sets = []
    for wl in water_levels:
        traces, annotations = generate_traces(wl)
        all_trace_sets.append(traces)
        all_annotation_sets.append(annotations)
    
    # Create figure
    fig = go.Figure(data=all_trace_sets[initial_wl_index])
    
    # Create slider steps
    slider_steps = []
    for i, wl in enumerate(water_levels):
        step = {
            'args': [
                {'x': [trace.x for trace in all_trace_sets[i]],
                 'y': [trace.y for trace in all_trace_sets[i]]},
                {'title': f'Poretrykkprofil - Vannstand: {wl:.2f} m',
                 'annotations': all_annotation_sets[i]}
            ],
            'label': f'{wl:.2f}',
            'method': 'update'
        }
        slider_steps.append(step)
    
    fig.update_layout(
        title=f"{borepoint_nr} - Poretrykkprofil - Vannstand: {water_level:.2f} m",
        xaxis_title="Trykk (kPa)",
        yaxis_title="Høyde (moh)",
        width=1123,
        height=794,
        annotations=all_annotation_sets[initial_wl_index],
        sliders=[{
            'active': initial_wl_index,
            'yanchor': 'top',
            'y': -0.15,
            'xanchor': 'left',
            'currentvalue': {
                'prefix': 'Vannstand: ',
                'suffix': ' m',
                'visible': True,
                'xanchor': 'center',
                'font': {'size': 14}
            },
            'pad': {'b': 10, 't': 50},
            'len': 0.9,
            'x': 0.05,
            'steps': slider_steps
        }]
    )
    
    # Save figure
    output_dir = Path(pvt_path_1).parent
    if has_second_sensor:
        output_file_html = output_dir / f"{serial_number_1}_{serial_number_2}_poretrykksprofil.html"
        output_file_pdf = output_dir / f"{serial_number_1}_{serial_number_2}_poretrykksprofil.pdf"
    else:
        output_file_html = output_dir / f"{serial_number_1}_poretrykksprofil.html"
        output_file_pdf = output_dir / f"{serial_number_1}_poretrykksprofil.pdf"
    
    fig.write_html(str(output_file_html))
    fig.show()
    # Try to save as PDF
    try:
        import arcpy
        has_arcpy = True
    except ImportError:
        has_arcpy = False
    
    try:
        fig.write_image(str(output_file_pdf), width=1123, height=794, scale=1)
        if output_file_pdf.exists():
            if has_arcpy:
                arcpy.AddMessage(f"PDF saved to: {output_file_pdf}")
    except Exception as e:
        if has_arcpy:
            arcpy.AddWarning(f"PDF generation failed: {str(e)}")
    
    return str(output_file_html)

#example usage of the function
process_poretrykksprofil(
    pvt_path_1=r"C:\Users\jdr\OneDrive - Multiconsult\Skrivebord\Test_PZ_plot\fv109 - Copy.xlsx"
    , serial_number_1="39434"
    , pvt_path_2=r"C:\Users\jdr\OneDrive - Multiconsult\Skrivebord\Test_PZ_plot\fv109 - Copy.xlsx"
    , serial_number_2="39433"
    , station_id="SN17000"
    , terrain_level=25
    , installation_depth_1=6
    , installation_depth_2=15
    , water_depth_below_terrain=3
    , borepoint_nr="Borepunkt A"
                )

'C:\\Users\\jdr\\OneDrive - Multiconsult\\Skrivebord\\Test_PZ_plot\\39434_39433_poretrykksprofil.html'